In [5]:
!pip install -q google-genai

In [6]:
from google import genai
from google.genai import types
from google.colab import userdata
import json

# userdata.get() reads the Secret you just added — this is Colab's
# equivalent of load_dotenv() reading a .env file.
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

MODEL = "gemini-2.5-flash"  # fast + free-tier friendly

In [7]:
PURCHASE_ORDERS = {
    "PO-1001": {"po_id": "PO-1001", "vendor": "Acme Packaging Co",
                "line_items": [{"item": "Corrugated boxes, 12x12", "quantity": 500, "unit_price": 2.10}]},
    "PO-1002": {"po_id": "PO-1002", "vendor": "Northwind Steel Supply",
                "line_items": [{"item": "Steel rod, 10mm", "quantity": 200, "unit_price": 14.50}]},
}

GOODS_RECEIPTS = {
    "PO-1001": {"po_id": "PO-1001", "quantity_received": 500},
    "PO-1002": {"po_id": "PO-1002", "quantity_received": 180},  # short receipt — the exception
}

INVOICES = {
    "INV-9001": {"invoice_id": "INV-9001", "po_id": "PO-1001", "vendor": "Acme Packaging Co",
                 "quantity_billed": 500, "unit_price": 2.10},
    "INV-9002": {"invoice_id": "INV-9002", "po_id": "PO-1002", "vendor": "Northwind Steel Supply",
                 "quantity_billed": 200, "unit_price": 14.50},  # billed 200, only 180 received
}

def get_purchase_order(po_id: str) -> dict:
    return PURCHASE_ORDERS.get(po_id, {"error": f"No PO found for {po_id}"})

def get_goods_receipt(po_id: str) -> dict:
    return GOODS_RECEIPTS.get(po_id, {"error": f"No goods receipt found for {po_id}"})

In [8]:
# --- TOOL DEFINITIONS ---
# These don't run anything. They're just DESCRIPTIONS of the two
# functions above, written in a schema Gemini understands, so the
# model knows: this tool exists, here's what it does, here's what
# argument it needs, and here's the argument's type.

get_po_tool = types.FunctionDeclaration(
    name="get_purchase_order",
    description="Fetch a purchase order's details by its PO ID.",
    parameters={
        "type": "OBJECT",
        "properties": {
            "po_id": {"type": "STRING", "description": "e.g. PO-1001"}
        },
        "required": ["po_id"],
    },
)

get_gr_tool = types.FunctionDeclaration(
    name="get_goods_receipt",
    description="Fetch the goods receipt (quantity actually received) for a PO ID.",
    parameters={
        "type": "OBJECT",
        "properties": {
            "po_id": {"type": "STRING", "description": "e.g. PO-1001"}
        },
        "required": ["po_id"],
    },
)

# Gemini expects tools grouped into a single "Tool" object, even
# when you're only handing it two of them.
tools = types.Tool(function_declarations=[get_po_tool, get_gr_tool])

# A plain Python dictionary mapping the NAME Gemini will say
# ("get_purchase_order") to the ACTUAL function we defined earlier.
# This is what lets our code translate "Gemini asked for X" into
# "run the real Python function X."
AVAILABLE_TOOLS = {
    "get_purchase_order": get_purchase_order,
    "get_goods_receipt": get_goods_receipt,
}

In [9]:
DECISION_SCHEMA = {
    "type": "OBJECT",
    "properties": {
        "variance_type": {"type": "STRING", "enum": ["quantity_mismatch", "price_mismatch", "no_variance", "insufficient_data"]},
        "evidence": {"type": "STRING"},
        "confidence": {"type": "NUMBER"},
        "recommended_action": {"type": "STRING", "enum": ["auto_resolve", "route_to_human_approval", "request_more_info"]},
    },
    "required": ["variance_type", "evidence", "confidence", "recommended_action"],
}

In [10]:
def diagnose_exception(invoice_id: str) -> dict:
    invoice = INVOICES[invoice_id]

    # The model gets ONLY the invoice — it must call tools to get the
    # PO and goods receipt itself. That's the whole "agent" behavior.
    contents = [
        types.Content(role="user", parts=[types.Part.from_text(
            text=f"Diagnose this invoice exception. Use the available tools to "
                 f"fetch the PO and goods receipt before concluding anything:\n"
                 f"{json.dumps(invoice, indent=2)}"
        )])
    ]

    # Tool-calling loop: keep going while the model keeps asking for tools.
    for _ in range(5):
        response = client.models.generate_content(
            model=MODEL,
            contents=contents,
            config=types.GenerateContentConfig(tools=[tools]),
        )

        calls = response.function_calls
        if not calls:
            break  # model is done gathering data

        # Record the model's own turn (its tool request) in the conversation...
        contents.append(response.candidates[0].content)

        # ...then run each requested tool and hand the result back.
        result_parts = []
        for call in calls:
            result = AVAILABLE_TOOLS[call.name](**call.args)
            result_parts.append(types.Part.from_function_response(name=call.name, response={"result": result}))
        contents.append(types.Content(role="user", parts=result_parts))

    # Final call: force the structured decision shape.
    final = client.models.generate_content(
        model=MODEL,
        contents=contents + [types.Content(role="user", parts=[types.Part.from_text(
            text="Now give your final diagnosis in the required structured format."
        )])],
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=DECISION_SCHEMA,
        ),
    )
    return json.loads(final.text)

In [11]:
import json

for inv_id in ["INV-9001", "INV-9002"]:
    print(f"\n{'='*50}\nDiagnosing {inv_id}\n{'='*50}")
    print(json.dumps(diagnose_exception(inv_id), indent=2))


Diagnosing INV-9001
{
  "variance_type": "no_variance",
  "evidence": "Invoice quantity billed (500) matches PO quantity (500) and goods received quantity (500). Invoice unit price ($2.1) matches PO unit price ($2.1).",
  "confidence": 1.0,
  "recommended_action": "auto_resolve"
}

Diagnosing INV-9002


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')